# 1. Setup & Imports

In [ ]:
import os, sys, json, torch, penman, numpy as np, pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader


print(f"Pytorch Version : {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Default Device : {device}")

# 2. Load Dataset

In [ ]:
# Absolute input path; change this line if the dataset moves.
XLSUM_DATASET_PATH = r"D:\Github\generate_amr\data\xlsum\analysis_data.csv"

class XLSumDataset(Dataset):
    """Custom Dataset For XLSum Only"""

    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return {
            # here id is use so we don't lose track the data and can be used
            # as name of the file if we want to store all the amr graph on .txt
            "id" : row["id"],
            "text" : row["text"],
        }

In [ ]:
df = pd.read_csv(XLSUM_DATASET_PATH)
ds = XLSumDataset(df)

# 3. Load Tokenizer & Model

In [ ]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import snapshot_download

# Download model locally first to bypass transformers/huggingface_hub version mismatch
nllb_local_path = snapshot_download("facebook/nllb-200-distilled-1.3B")
print(f"NLLB model downloaded to: {nllb_local_path}")

nllb_tokenizer = NllbTokenizer.from_pretrained(nllb_local_path)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_local_path).to(device)
print(f"NLLB model loaded successfully!")

print(f"Model Loaded on : {nllb_model.device}")
print(f"Model Parameters : {sum([p.numel() for p in nllb_model.parameters() if p.requires_grad is True])}")

# 4. Create Data Loader

In [ ]:
dl = DataLoader(ds, batch_size=4)

In [ ]:
from pathlib import Path

def save_file(folder_path : str):
    os.makedirs(folder_path, exist_ok=True)

    def process(filename : str, translated : str):
        with open(os.path.join(folder_path, filename), "w") as f:
            f.write(translated)

    def inner_fn(filename : str, translated : str):
        try:
            process(filename, translated)
            tqdm.write(f"Successfully write translatd file for {filename}")
        except Exception as e:
            tqdm.write("Something went wrong")
            tqdm.write(e)

    return inner_fn

def is_file_exist(file_path : str):
    file_path = Path(file_path)

    if file_path.is_file():
        return True
    return False

# 5. Translate

In [ ]:
OUTPUT_PATH = r"D:\Github\generate_amr\data\xlsum\translate"
save_file_fn = save_file(OUTPUT_PATH)
for batch in tqdm(dl):
    ids, texts = batch["id"], batch["text"]
    masking = []

    for idx, id in enumerate(ids):
        if is_file_exist(os.path.join(OUTPUT_PATH, f"{id}.txt")):
            tqdm.write(f"File {id}.txt already exist. Skipping file...")
            continue
        masking.append(idx)

    if len(masking) == 0:
        continue

    ids, texts = [ids[i] for i in masking], [texts[i] for i in masking]
    
    tokenized_texts = nllb_tokenizer(
        texts, padding = True, truncation = True, max_length = None,
        return_tensors="pt"
    ).to(nllb_model.device)
    
    translated_tokens = nllb_model.generate(
        input_ids = tokenized_texts["input_ids"],
        attention_mask = tokenized_texts["attention_mask"],
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn")
    )

    translated_texts = nllb_tokenizer.batch_decode(translated_tokens,
                                                    skip_special_tokens=True)
    
    for id, text in zip(ids, texts):
        save_file_fn(f"{id}.txt", text)

In [ ]:
nllb_tokenizer.batch_decode(translated_texts, skip_special_tokens=True)